# Sell Smart AI — Final Decision System

This notebook combines the **three existing models without modifying or retraining them**:

1. **Model 1 — Ethiopia Wheat Price Model:** predicts the future wheat price.
2. **Model 2 — Storage Loss Model:** estimates expected storage loss using the saved region-smoothed artifact.
3. **Model 3 — Transport Cost Model:** predicts/estimates transport cost.

The decision layer then compares selling now with storing and selling later.

**Final user-facing outputs:** current price, predicted price, storage loss, transport cost, financial comparison, final decision, and recommendation.

> Important: keep the original `.joblib` model files unchanged. Upload the three model files to Colab when prompted.

In [ ]:
# Install the sklearn version used by the price model.
# Do not change the saved model files.
!pip -q install scikit-learn==1.6.1 joblib pandas numpy


In [ ]:
from google.colab import files
import os, json, math
import joblib
import numpy as np
import pandas as pd

print('Upload these three files:')
print('1) ethiopia_wheat_price_model_FINAL (1).joblib')
print('2) model_2_storage_loss_V2_COMPLETE.joblib')
print('3) Model_3_Transport_Cost_V4_FINAL.joblib')

uploaded = files.upload()
print('\nUploaded:', list(uploaded.keys()))


In [ ]:
# Locate the three uploaded model files.
def find_file(fragment):
    matches = [f for f in os.listdir('.') if fragment.lower() in f.lower()]
    if not matches:
        raise FileNotFoundError(f'Could not find a file containing: {fragment}')
    return matches[0]

PRICE_MODEL_FILE = find_file('ethiopia_wheat_price_model')
STORAGE_MODEL_FILE = find_file('model_2_storage_loss')
TRANSPORT_MODEL_FILE = find_file('Model_3_Transport_Cost')

print('Price model   :', PRICE_MODEL_FILE)
print('Storage model :', STORAGE_MODEL_FILE)
print('Transport model:', TRANSPORT_MODEL_FILE)


In [ ]:
# Load the artifacts exactly as saved.
price_artifact = joblib.load(PRICE_MODEL_FILE)
storage_artifact = joblib.load(STORAGE_MODEL_FILE)
transport_artifact = joblib.load(TRANSPORT_MODEL_FILE)

def describe(obj, name):
    print(f'\n{name}')
    print('type:', type(obj))
    if isinstance(obj, dict):
        print('keys:', list(obj.keys()))
    else:
        print('object:', obj)

describe(price_artifact, 'PRICE ARTIFACT')
describe(storage_artifact, 'STORAGE ARTIFACT')
describe(transport_artifact, 'TRANSPORT ARTIFACT')


## 1. Model 1 — Price prediction adapter

The adapter below is intentionally flexible so the saved price artifact can be used without altering it. It first checks common artifact structures and then uses the saved model's own feature names when available.

**For production:** use the same feature names and preprocessing that were used when Model 1 was trained.

In [ ]:
def unwrap_estimator(artifact):
    """Find a fitted sklearn-like estimator inside a common saved-artifact structure."""
    if hasattr(artifact, 'predict'):
        return artifact
    if isinstance(artifact, dict):
        preferred = ['model', 'estimator', 'pipeline', 'regressor', 'price_model']
        for key in preferred:
            value = artifact.get(key)
            if hasattr(value, 'predict'):
                return value
    return None

price_model = unwrap_estimator(price_artifact)
transport_model = unwrap_estimator(transport_artifact)

print('Price estimator:', type(price_model))
print('Transport estimator:', type(transport_model))


## 2. Model 2 — Storage-loss calculation

The COMPLETE Model 2 artifact is a **region-smoothed expected-loss model**. It is not a normal sklearn estimator with `predict()`.

Its calculation uses the saved region statistics, global mean, and smoothing alpha. This cell reads those values directly from the artifact.

In [ ]:
def storage_loss_prediction(region, artifact=storage_artifact):
    if not isinstance(artifact, dict):
        raise TypeError('Expected the Model 2 COMPLETE artifact to be a dictionary.')
        
    region_stats = artifact.get('region_stats', {})
    global_mean = float(artifact.get('global_mean', 0.0))
    alpha = float(artifact.get('region_smoothing_alpha', 50.0))
        
    # Support either a direct mapping or a DataFrame-like region_stats object.
    stats = None
    if isinstance(region_stats, dict):
        stats = region_stats.get(region)
        if stats is None:
            stats = region_stats.get(str(region))
    
    if stats is None:
        return global_mean, 'global_mean_fallback'
    
    if isinstance(stats, dict):
        region_mean = stats.get('mean', stats.get('loss_mean', stats.get('storage_loss_mean')))
        region_count = stats.get('count', stats.get('n', stats.get('region_count')))
    elif isinstance(stats, (list, tuple)) and len(stats) >= 2:
        region_mean, region_count = stats[0], stats[1]
    else:
        region_mean = None
        region_count = None
    
    if region_mean is None or region_count is None:
        return global_mean, 'global_mean_fallback'
    
    smoothed = (float(region_mean) * float(region_count) + global_mean * alpha) / (float(region_count) + alpha)
    return smoothed, 'region_smoothed'

print('Model 2 type:', storage_artifact.get('model_type') if isinstance(storage_artifact, dict) else type(storage_artifact))
print('Global mean:', storage_artifact.get('global_mean') if isinstance(storage_artifact, dict) else 'n/a')
print('Smoothing alpha:', storage_artifact.get('region_smoothing_alpha') if isinstance(storage_artifact, dict) else 'n/a')


## 3. Model 3 — Transport-cost adapter

The transport adapter uses the saved estimator when it exposes `predict()`. The exact feature dictionary for Model 3 is supplied by the user/application and converted into a one-row DataFrame using the model's saved feature names where available.

In [ ]:
def estimator_feature_names(estimator):
    if estimator is None:
        return None
    for attr in ['feature_names_in_']:
        names = getattr(estimator, attr, None)
        if names is not None:
            return list(names)
    return None

print('Transport feature names:', estimator_feature_names(transport_model))


## 4. Unified Sell Smart AI decision function

This is the part that the web app will eventually call.

The decision system returns **predicted price explicitly**, along with storage loss, transport cost, financial comparison, decision, reasons, and recommendations.

In [ ]:
def predict_price(price_features):
    if price_model is None:
        raise RuntimeError('No sklearn-like price estimator with predict() was found in Model 1.')
    
    feature_names = estimator_feature_names(price_model)
    if feature_names is not None:
        missing = [f for f in feature_names if f not in price_features]
        if missing:
            raise ValueError(f'Missing Model 1 features: {missing}')
        X = pd.DataFrame([{f: price_features[f] for f in feature_names}])
    else:
        X = pd.DataFrame([price_features])
    
    return float(np.asarray(price_model.predict(X)).reshape(-1)[0])

def predict_transport(transport_features):
    if transport_model is None:
        raise RuntimeError('No sklearn-like transport estimator with predict() was found in Model 3.')
    
    feature_names = estimator_feature_names(transport_model)
    if feature_names is not None:
        missing = [f for f in feature_names if f not in transport_features]
        if missing:
            raise ValueError(f'Missing Model 3 features: {missing}')
        X = pd.DataFrame([{f: transport_features[f] for f in feature_names}])
    else:
        X = pd.DataFrame([transport_features])
    
    return float(np.asarray(transport_model.predict(X)).reshape(-1)[0])

def sell_smart_decision(
    current_price,
    quantity,
    region,
    price_features,
    transport_features,
    transport_cost_now=None,
    transport_cost_later=None,
    storage_cost_per_unit=0.0,
    storage_horizon_days=30,
    decision_threshold_pct=2.0,
):
    """Return the complete Sell Smart AI result."""
    current_price = float(current_price)
    quantity = float(quantity)
    
    predicted_price = predict_price(price_features)
    storage_loss_pct, storage_status = storage_loss_prediction(region)
    
    if transport_cost_now is None:
        transport_cost_now = predict_transport(transport_features)
    if transport_cost_later is None:
        transport_cost_later = float(transport_cost_now)
    
    transport_cost_now = float(transport_cost_now)
    transport_cost_later = float(transport_cost_later)
    
    # Model 2 is expressed as a percentage loss.
    loss_fraction = max(0.0, storage_loss_pct) / 100.0
    sale_quantity_later = quantity * (1.0 - loss_fraction)
    
    sell_now_value = quantity * current_price - transport_cost_now
    store_then_sell_value = sale_quantity_later * predicted_price - transport_cost_later - quantity * float(storage_cost_per_unit)
    advantage = store_then_sell_value - sell_now_value
    base_value = max(abs(sell_now_value), 1e-9)
    advantage_pct = 100.0 * advantage / base_value
    
    # Transparent decision rule.
    if advantage_pct >= decision_threshold_pct:
        decision = 'STORE'
    elif advantage_pct <= -decision_threshold_pct:
        decision = 'SELL_NOW'
    else:
        decision = 'STORE_CAUTION'
    
    reasons = [
        f'Predicted future price: {predicted_price:,.2f} ETB per unit.',
        f'Expected storage loss: {storage_loss_pct:.2f}%.',
        f'Selling now value after current transport: {sell_now_value:,.2f} ETB.',
        f'Store-then-sell estimated value: {store_then_sell_value:,.2f} ETB.',
        f'Estimated advantage of storing: {advantage:,.2f} ETB ({advantage_pct:.2f}%).'
    ]
    
    if decision == 'STORE':
        recommendation = 'Store the wheat if the real storage conditions match the estimate, then consider selling as the market approaches the predicted price.'
    elif decision == 'SELL_NOW':
        recommendation = 'Selling now is financially preferable under the model assumptions; avoid unnecessary storage costs and losses.'
    else:
        recommendation = 'The difference is small. Compare actual local offers, storage conditions, and transport costs before deciding.'
    
    return {
        'current_price': current_price,
        'predicted_price': predicted_price,
        'expected_storage_loss_pct': float(storage_loss_pct),
        'storage_status': storage_status,
        'transport_cost_now': transport_cost_now,
        'transport_cost_later': transport_cost_later,
        'quantity': quantity,
        'expected_quantity_after_storage': sale_quantity_later,
        'sell_now_value': sell_now_value,
        'store_then_sell_value': store_then_sell_value,
        'expected_financial_advantage': advantage,
        'expected_financial_advantage_pct': advantage_pct,
        'decision': decision,
        'recommendation': recommendation,
        'reasons': reasons,
    }


## 5. Test the complete system

Replace the example values below with the **same feature names and units used by your three models**. The cell will print the exact information that the web app should display.

In [ ]:
# IMPORTANT: fill these dictionaries using the feature names printed by your saved models.
# Example structure only — do not submit these placeholder values as real predictions.

PRICE_FEATURES = {
    # 'feature_name_from_model_1': value,
}

TRANSPORT_FEATURES = {
    # 'feature_name_from_model_3': value,
}

# Example:
# result = sell_smart_decision(
#     current_price=30000,
#     quantity=100,
#     region='Oromia',
#     price_features=PRICE_FEATURES,
#     transport_features=TRANSPORT_FEATURES,
#     storage_cost_per_unit=0,
# )
# print(json.dumps(result, indent=2))


## 6. Production output format

The web app should expose these fields:

- `current_price`
- `predicted_price` ⭐
- `expected_storage_loss_pct`
- `transport_cost_now`
- `transport_cost_later`
- `sell_now_value`
- `store_then_sell_value`
- `expected_financial_advantage`
- `decision`
- `recommendation`
- `reasons`

This keeps prediction and decision separate: **Model 1 predicts the price; the decision system decides what to do with that prediction.**